# Simulation Analysis of Downstream Performance-Estimation Error

**Goal**: Compare how well CT-RATE original labels vs LLM majority-vote labels estimate downstream model performance, using radiologist annotations as the true ground truth.

**Design**:
1. Synthetic prediction labels are generated to approximate pre-specified sensitivity/specificity against radiologist GT.
2. Synthetic predictions are evaluated using three reference standards: radiologist GT, CT-RATE, and LLM majority vote.
3. Absolute deviation from radiologist-GT-based estimates is the primary outcome.
4. Uncertainty is assessed via report-level bootstrap resampling (B = 1000).

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

DATA_DIR    = Path('../data')
FIGURES_DIR = Path('../figures')
TABLES_DIR  = Path('../tables')

LABEL_COLS = [
    'Medical material',
    'Arterial wall calcification',
    'Cardiomegaly',
    'Pericardial effusion',
    'Coronary artery wall calcification',
    'Hiatal hernia',
    'Lymphadenopathy',
    'Emphysema',
    'Atelectasis',
    'Lung nodule',
    'Lung opacity',
    'Pulmonary fibrotic sequela',
    'Pleural effusion',
    'Mosaic attenuation pattern',
    'Peribronchial thickening',
    'Consolidation',
    'Bronchiectasis',
    'Interlobular septal thickening',
]

LLM_MODELS = ['GPT-5.4', 'GPT-5.4 mini', 'DeepSeek-V3.2']

SCENARIOS = {
    'high_balanced':      {'sensitivity': 0.90, 'specificity': 0.90},
    'high_sensitivity':   {'sensitivity': 0.95, 'specificity': 0.85},
    'high_specificity':   {'sensitivity': 0.85, 'specificity': 0.95},
    'moderate_balanced':  {'sensitivity': 0.80, 'specificity': 0.80},
    'weak_balanced':      {'sensitivity': 0.70, 'specificity': 0.70},
}

SCENARIO_LABELS = {
    'high_balanced':      'High balanced\n(Sen=0.90, Spe=0.90)',
    'high_sensitivity':   'High sensitivity\n(Sen=0.95, Spe=0.85)',
    'high_specificity':   'High specificity\n(Sen=0.85, Spe=0.95)',
    'moderate_balanced':  'Moderate balanced\n(Sen=0.80, Spe=0.80)',
    'weak_balanced':      'Weak balanced\n(Sen=0.70, Spe=0.70)',
}



SCENARIO_SHORT = {
    'high_balanced':      'High balanced',
    'high_sensitivity':   'High sensitivity',
    'high_specificity':   'High specificity',
    'moderate_balanced':  'Moderate balanced',
    'weak_balanced':      'Weak balanced',
}

REFERENCES = ['radiologist_gt', 'ct_rate', 'llm_vote']
REF_LABELS  = {
    'radiologist_gt': 'Radiologist GT',
    'ct_rate':        'CT-RATE (original)',
    'llm_vote':       'LLM majority vote',
}
REF_COLORS = {
    'radiologist_gt': '#2c7bb6',
    'ct_rate':        '#d7191c',
    'llm_vote':       '#1a9641',
}

METRICS = ['Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1', 'Balanced_accuracy']

B           = 1000
RANDOM_SEED = 42

print('Setup complete.')
print(f'Scenarios : {list(SCENARIOS.keys())}')
print(f'Bootstrap : B={B}, seed={RANDOM_SEED}')

## 1. Load Data and Build Long-Format Table

In [ ]:
# --- Radiologist GT (random_annotations.csv) ---
ann_long = pd.read_csv(DATA_DIR / 'random_annotations.csv')
human = (
    ann_long
    .pivot_table(index='volume_name', columns='label_name', values='value', aggfunc='first')
    .reset_index()
    .rename(columns={'volume_name': 'VolumeName'})
)
human = human[['VolumeName'] + LABEL_COLS]
annotated_volumes = set(human['VolumeName'])
print(f'Radiologist GT: {len(human)} reports, {len(LABEL_COLS)} labels')

# --- CT-RATE original labels ---
ctrate_wide = pd.read_csv(DATA_DIR / 'CT-RATE_reports.csv')
ctrate_wide = ctrate_wide[ctrate_wide['VolumeName'].isin(annotated_volumes)][['VolumeName'] + LABEL_COLS]
print(f'CT-RATE original: {len(ctrate_wide)} reports matched')

# --- LLM predictions → majority vote ---
SOURCE_FILES = {
    'GPT-5.4':       DATA_DIR / 'CT-RATE_reports_gpt-5.4.csv',
    'GPT-5.4 mini':  DATA_DIR / 'CT-RATE_reports_gpt-5.4-mini.csv',
    'DeepSeek-V3.2': DATA_DIR / 'CT-RATE_reports_DeepSeek-V3.2.csv',
}

llm_dfs = {}
for name, path in SOURCE_FILES.items():
    df = pd.read_csv(path)
    if '_extraction_failed' in df.columns:
        df = df[~df['_extraction_failed'].fillna(True).astype(bool)].copy()
    df = df[df['VolumeName'].isin(annotated_volumes)][['VolumeName'] + LABEL_COLS].copy()
    llm_dfs[name] = df
    print(f'{name}: {len(df)} reports matched')

# Find volumes present in ALL sources
common_volumes = annotated_volumes.copy()
for df in [ctrate_wide] + list(llm_dfs.values()):
    common_volumes &= set(df['VolumeName'])
print(f'\nCommon volumes (all sources): {len(common_volumes)}')

SORTED_IDX = sorted(common_volumes)

human_idx  = human.set_index('VolumeName').loc[SORTED_IDX]
ctrate_idx = ctrate_wide.set_index('VolumeName').loc[SORTED_IDX]
llm_idx    = {name: df.set_index('VolumeName').loc[SORTED_IDX] for name, df in llm_dfs.items()}

# Majority vote: positive if ≥2/3 LLMs agree
vote_df = pd.DataFrame(index=SORTED_IDX)
for col in LABEL_COLS:
    votes = np.stack([llm_idx[m][col].astype(int).values for m in LLM_MODELS], axis=1)
    vote_df[col] = (votes.sum(axis=1) >= 2).astype(int)
print('Majority vote computed (≥2/3 models positive)')

In [ ]:
# Build long-format table: report_id | label_name | radiologist_gt | ct_rate | llm_vote
rows = []
for vol in SORTED_IDX:
    for lbl in LABEL_COLS:
        rows.append({
            'report_id':      vol,
            'label_name':     lbl,
            'radiologist_gt': int(human_idx.loc[vol, lbl]),
            'ct_rate':        int(ctrate_idx.loc[vol, lbl]),
            'llm_vote':       int(vote_df.loc[vol, lbl]),
        })

long_df = pd.DataFrame(rows)
print(f'Long-format table: {len(long_df):,} rows ({len(SORTED_IDX)} reports × {len(LABEL_COLS)} labels)')
print()
print('Preview:')
print(long_df.head(8).to_string(index=False))
print()
print('Positive rates:')
print(f'  Radiologist GT : {long_df["radiologist_gt"].mean():.3f}')
print(f'  CT-RATE        : {long_df["ct_rate"].mean():.3f}')
print(f'  LLM majority   : {long_df["llm_vote"].mean():.3f}')

## 2. Synthetic Prediction Generation

For each scenario and each label, synthetic predictions are generated once and held fixed.

- Positives (`gt == 1`): assign `pred = 1` to `round(n_pos * sensitivity)` cases.
- Negatives (`gt == 0`): assign `pred = 0` to `round(n_neg * specificity)` cases.

Due to integer rounding, achieved sensitivity/specificity may differ slightly from the target.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

achieved_rows = []

for scenario, params in SCENARIOS.items():
    target_sens = params['sensitivity']
    target_spec = params['specificity']
    pred_col    = f'pred_{scenario}'
    long_df[pred_col] = 0

    for lbl in LABEL_COLS:
        mask_lbl = long_df['label_name'] == lbl
        pos_idx  = long_df.index[mask_lbl & (long_df['radiologist_gt'] == 1)].tolist()
        neg_idx  = long_df.index[mask_lbl & (long_df['radiologist_gt'] == 0)].tolist()

        n_pos = len(pos_idx)
        n_neg = len(neg_idx)

        # Positives: select round(n_pos * sensitivity) as pred=1
        n_tp = int(round(n_pos * target_sens))
        if n_pos > 0:
            tp_idx = rng.choice(pos_idx, size=n_tp, replace=False).tolist()
            long_df.loc[tp_idx, pred_col] = 1

        # Negatives: select round(n_neg * (1 - specificity)) as pred=1 (false positives)
        n_fp = int(round(n_neg * (1 - target_spec)))
        if n_neg > 0 and n_fp > 0:
            fp_idx = rng.choice(neg_idx, size=n_fp, replace=False).tolist()
            long_df.loc[fp_idx, pred_col] = 1

        # Compute achieved metrics
        sub = long_df[mask_lbl]
        achieved_sens = (sub[pred_col] & (sub['radiologist_gt'] == 1)).sum() / n_pos if n_pos > 0 else np.nan
        achieved_spec = (~sub[pred_col].astype(bool) & (sub['radiologist_gt'] == 0)).sum() / n_neg if n_neg > 0 else np.nan

        achieved_rows.append({
            'scenario':             scenario,
            'label_name':           lbl,
            'n_pos':                n_pos,
            'n_neg':                n_neg,
            'target_sensitivity':   target_sens,
            'target_specificity':   target_spec,
            'achieved_sensitivity': round(achieved_sens, 4) if not np.isnan(achieved_sens) else np.nan,
            'achieved_specificity': round(achieved_spec, 4) if not np.isnan(achieved_spec) else np.nan,
        })

achieved_df = pd.DataFrame(achieved_rows)

print('Synthetic predictions generated.')
print()
print('Achieved sensitivity/specificity vs target (mean across 18 labels):')
summary = (
    achieved_df
    .groupby('scenario')[['target_sensitivity', 'achieved_sensitivity',
                           'target_specificity', 'achieved_specificity']]
    .mean()
    .round(3)
)
print(summary.to_string())

In [ ]:
# Save achieved sensitivity/specificity table
achieved_df.to_csv(TABLES_DIR / 'sim_achieved_sens_spec.csv', index=False)
print('Saved: sim_achieved_sens_spec.csv')
print()
print('Achieved sensitivity/specificity — full table (sample):')
print(achieved_df[achieved_df['scenario'] == 'high_balanced'].to_string(index=False))

## 3. Performance Evaluation Functions

In [ ]:
def compute_metrics_from_cm(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute all metrics from binary arrays. Returns dict with NaN for undefined metrics."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = len(y_true)

    sensitivity       = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity       = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv               = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv               = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    accuracy          = (tp + tn) / n
    f1                = f1_score(y_true, y_pred, average='binary', zero_division=0)
    balanced_accuracy = np.nanmean([sensitivity, specificity])

    return {
        'Accuracy':          float(accuracy),
        'Sensitivity':       float(sensitivity) if not np.isnan(sensitivity) else np.nan,
        'Specificity':       float(specificity) if not np.isnan(specificity) else np.nan,
        'PPV':               float(ppv)         if not np.isnan(ppv)         else np.nan,
        'NPV':               float(npv)         if not np.isnan(npv)         else np.nan,
        'F1':                float(f1),
        'Balanced_accuracy': float(balanced_accuracy),
    }


def evaluate_scenario_on_df(sub_df: pd.DataFrame, scenario: str) -> dict:
    """
    Evaluate scenario predictions against all three reference standards
    on a (possibly bootstrapped) subset dataframe.
    Returns metrics for each reference as flat dict.
    """
    pred_col = f'pred_{scenario}'
    y_pred   = sub_df[pred_col].astype(int).values
    results  = {}
    for ref in REFERENCES:
        y_ref = sub_df[ref].astype(int).values
        m = compute_metrics_from_cm(y_ref, y_pred)
        for metric_name, val in m.items():
            results[f'{ref}__{metric_name}'] = val
    return results


print('Metric functions defined.')

## 4. Point Estimates (Full Dataset)

Evaluate each scenario against all three reference standards using all 1,800 label instances.

In [ ]:
point_rows = []
for scenario in SCENARIOS:
    res = evaluate_scenario_on_df(long_df, scenario)
    for ref in REFERENCES:
        for metric in METRICS:
            point_rows.append({
                'scenario':  scenario,
                'reference': ref,
                'metric':    metric,
                'estimate':  res[f'{ref}__{metric}'],
            })

point_df = pd.DataFrame(point_rows)

# Compute deviation from radiologist_gt
gt_vals = point_df[point_df['reference'] == 'radiologist_gt'][['scenario', 'metric', 'estimate']]
gt_vals = gt_vals.rename(columns={'estimate': 'gt_estimate'})

point_df = point_df.merge(gt_vals, on=['scenario', 'metric'])
point_df['abs_deviation'] = (point_df['estimate'] - point_df['gt_estimate']).abs()

# Deviation table: ct_rate vs llm_vote side by side
dev_ct  = point_df[point_df['reference'] == 'ct_rate'][['scenario', 'metric', 'abs_deviation']].rename(columns={'abs_deviation': 'ct_rate_error'})
dev_llm = point_df[point_df['reference'] == 'llm_vote'][['scenario', 'metric', 'abs_deviation']].rename(columns={'abs_deviation': 'llm_vote_error'})

dev_df = dev_ct.merge(dev_llm, on=['scenario', 'metric'])
dev_df['error_reduction'] = dev_df['ct_rate_error'] - dev_df['llm_vote_error']

print('Point estimates computed.')
print()
print('Performance estimates (F1 and Balanced accuracy):')
pivot_disp = point_df[point_df['metric'].isin(['F1', 'Balanced_accuracy'])].pivot_table(
    index=['scenario', 'metric'], columns='reference', values='estimate'
)[REFERENCES]
pivot_disp.columns = [REF_LABELS[r] for r in REFERENCES]
print(pivot_disp.round(4).to_string())

In [ ]:
print('Absolute deviation from radiologist-GT-based estimates (F1 and Balanced accuracy):')
print()
for metric in ['F1', 'Balanced_accuracy']:
    sub = dev_df[dev_df['metric'] == metric][['scenario', 'ct_rate_error', 'llm_vote_error', 'error_reduction']]
    sub = sub.set_index('scenario').round(4)
    print(f'Metric: {metric}')
    print(sub.to_string())
    print()

## 5. Report-Level Bootstrap (B = 1000)

Bootstrap procedure:
1. Sample 100 reports with replacement.
2. Include all 18 labels for each sampled report.
3. Apply fixed synthetic predictions.
4. Compute metrics against all three reference standards.
5. Repeat B times.

In [ ]:
rng_bs = np.random.default_rng(RANDOM_SEED)

report_ids = np.array(SORTED_IDX)
N_reports  = len(report_ids)

# Store bootstrap results per scenario × reference × metric
# Shape: B × n_scenarios × n_refs × n_metrics
bs_store = {
    scenario: {ref: {metric: [] for metric in METRICS} for ref in REFERENCES}
    for scenario in SCENARIOS
}

for b in range(B):
    # Sample reports with replacement
    sampled_reports = rng_bs.choice(report_ids, size=N_reports, replace=True)
    # Replicate rows for sampled reports (all 18 labels per report)
    sub_df = long_df[long_df['report_id'].isin(sampled_reports)].copy()
    # Weight by sampling multiplicity
    report_counts = pd.Series(sampled_reports).value_counts()
    sub_rows = []
    for rep_id, count in report_counts.items():
        rep_rows = long_df[long_df['report_id'] == rep_id]
        for _ in range(count):
            sub_rows.append(rep_rows)
    sub_df = pd.concat(sub_rows, ignore_index=True)

    for scenario in SCENARIOS:
        res = evaluate_scenario_on_df(sub_df, scenario)
        for ref in REFERENCES:
            for metric in METRICS:
                bs_store[scenario][ref][metric].append(res[f'{ref}__{metric}'])

print(f'Bootstrap complete. B={B} iterations done.')

In [ ]:
# Compute bootstrap CIs and assemble results table
bs_rows = []
for scenario in SCENARIOS:
    for ref in REFERENCES:
        for metric in METRICS:
            vals = np.array(bs_store[scenario][ref][metric])
            bs_rows.append({
                'scenario':         scenario,
                'reference':        ref,
                'metric':           metric,
                'bs_mean':          float(np.nanmean(vals)),
                'bs_median':        float(np.nanmedian(vals)),
                'bootstrap_lower':  float(np.nanpercentile(vals, 2.5)),
                'bootstrap_upper':  float(np.nanpercentile(vals, 97.5)),
            })

bs_df = pd.DataFrame(bs_rows)

# Merge with point estimates
results_df = point_df.merge(
    bs_df[['scenario', 'reference', 'metric', 'bootstrap_lower', 'bootstrap_upper']],
    on=['scenario', 'reference', 'metric']
)

print('Bootstrap CIs computed.')
print()
print('Main results table (F1, high_balanced scenario):')
sub = results_df[
    (results_df['metric'] == 'F1') & (results_df['scenario'] == 'high_balanced')
][['scenario', 'metric', 'reference', 'estimate', 'bootstrap_lower', 'bootstrap_upper']]
sub['reference'] = sub['reference'].map(REF_LABELS)
print(sub.to_string(index=False))

In [ ]:
# Bootstrap CIs for deviation + direct CT-RATE vs LLM comparison
dev_bs_rows = []
for scenario in SCENARIOS:
    for metric in METRICS:
        gt_vals_bs  = np.array(bs_store[scenario]['radiologist_gt'][metric])
        ct_vals_bs  = np.array(bs_store[scenario]['ct_rate'][metric])
        llm_vals_bs = np.array(bs_store[scenario]['llm_vote'][metric])

        ct_error_bs  = np.abs(ct_vals_bs  - gt_vals_bs)
        llm_error_bs = np.abs(llm_vals_bs - gt_vals_bs)
        reduction_bs = ct_error_bs - llm_error_bs
        diff_bs      = ct_vals_bs - llm_vals_bs   # CT-RATE estimate minus LLM vote estimate

        pt = dev_df[(dev_df['scenario'] == scenario) & (dev_df['metric'] == metric)]
        ct_err_pt  = float(pt['ct_rate_error'].values[0])  if len(pt) > 0 else np.nan
        llm_err_pt = float(pt['llm_vote_error'].values[0]) if len(pt) > 0 else np.nan
        red_pt     = float(pt['error_reduction'].values[0]) if len(pt) > 0 else np.nan

        ct_est  = point_df[(point_df['scenario'] == scenario) & (point_df['metric'] == metric) & (point_df['reference'] == 'ct_rate')]['estimate'].values
        llm_est = point_df[(point_df['scenario'] == scenario) & (point_df['metric'] == metric) & (point_df['reference'] == 'llm_vote')]['estimate'].values
        ct_vs_llm_pt = float(ct_est[0] - llm_est[0]) if (len(ct_est) > 0 and len(llm_est) > 0) else np.nan

        dev_bs_rows.append({
            'scenario':            scenario,
            'metric':              metric,
            'ct_rate_error':       ct_err_pt,
            'ct_rate_error_lo':    float(np.nanpercentile(ct_error_bs, 2.5)),
            'ct_rate_error_hi':    float(np.nanpercentile(ct_error_bs, 97.5)),
            'llm_vote_error':      llm_err_pt,
            'llm_vote_error_lo':   float(np.nanpercentile(llm_error_bs, 2.5)),
            'llm_vote_error_hi':   float(np.nanpercentile(llm_error_bs, 97.5)),
            'error_reduction':     red_pt,
            'error_reduction_lo':  float(np.nanpercentile(reduction_bs, 2.5)),
            'error_reduction_hi':  float(np.nanpercentile(reduction_bs, 97.5)),
            'ct_vs_llm':           ct_vs_llm_pt,
            'ct_vs_llm_lo':        float(np.nanpercentile(diff_bs, 2.5)),
            'ct_vs_llm_hi':        float(np.nanpercentile(diff_bs, 97.5)),
        })

dev_bs_df = pd.DataFrame(dev_bs_rows)

print('Deviation bootstrap CIs computed.')
print()
print('Deviation table (F1):')
sub = dev_bs_df[dev_bs_df['metric'] == 'F1'][
    ['scenario', 'metric', 'ct_rate_error', 'llm_vote_error', 'error_reduction',
     'error_reduction_lo', 'error_reduction_hi', 'ct_vs_llm', 'ct_vs_llm_lo', 'ct_vs_llm_hi']
].round(4)
print(sub.to_string(index=False))

## 6. Summary Tables

In [ ]:
# Table A: Performance estimates with 95% CI
tbl_a_rows = []
for scenario in SCENARIOS:
    for metric in ['F1', 'Balanced_accuracy', 'Sensitivity', 'Specificity']:
        for ref in REFERENCES:
            r = results_df[
                (results_df['scenario'] == scenario) &
                (results_df['metric']   == metric)   &
                (results_df['reference'] == ref)
            ]
            if len(r) == 0:
                continue
            tbl_a_rows.append({
                'Scenario':    scenario,
                'Metric':      metric,
                'Reference':   REF_LABELS[ref],
                'Estimate':    round(float(r['estimate'].values[0]), 4),
                'CI_lower':    round(float(r['bootstrap_lower'].values[0]), 4),
                'CI_upper':    round(float(r['bootstrap_upper'].values[0]), 4),
            })

tbl_a = pd.DataFrame(tbl_a_rows)
tbl_a.to_csv(TABLES_DIR / 'sim_performance_estimates.csv', index=False)
print('Saved: sim_performance_estimates.csv')
print()
print('Table A excerpt (F1):')
print(tbl_a[tbl_a['Metric'] == 'F1'].to_string(index=False))

In [ ]:
# Table B: Deviation from radiologist-GT
tbl_b_rows = []
for scenario in SCENARIOS:
    for metric in ['F1', 'Balanced_accuracy', 'Sensitivity', 'Specificity']:
        r = dev_bs_df[(dev_bs_df['scenario'] == scenario) & (dev_bs_df['metric'] == metric)]
        if len(r) == 0:
            continue
        row = r.iloc[0]
        tbl_b_rows.append({
            'Scenario':          scenario,
            'Metric':            metric,
            'CT-RATE error':     round(row['ct_rate_error'], 4),
            'CT-RATE 95% CI':    f'({row["ct_rate_error_lo"]:.4f}–{row["ct_rate_error_hi"]:.4f})',
            'LLM vote error':    round(row['llm_vote_error'], 4),
            'LLM vote 95% CI':   f'({row["llm_vote_error_lo"]:.4f}–{row["llm_vote_error_hi"]:.4f})',
            'Error reduction':   round(row['error_reduction'], 4),
            'Reduction 95% CI':  f'({row["error_reduction_lo"]:.4f}–{row["error_reduction_hi"]:.4f})',
        })

tbl_b = pd.DataFrame(tbl_b_rows)
tbl_b.to_csv(TABLES_DIR / 'sim_deviation_table.csv', index=False)
print('Saved: sim_deviation_table.csv')
print()
print('Table B — Deviation from radiologist-GT-based estimates:')
print(tbl_b.to_string(index=False))

## 7. Figures

### Figure 2A: Simulation Design Schematic

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

box_style = dict(boxstyle='round,pad=0.5', facecolor='#ddeeff', edgecolor='#336699', linewidth=1.5)
arrow_props = dict(arrowstyle='->', color='#336699', lw=1.5)

ax.text(5, 5.3, 'Radiologist-annotated\nreference set\n(100 reports × 18 labels)', ha='center', va='center',
        fontsize=10, bbox=dict(boxstyle='round,pad=0.5', facecolor='#cce5ff', edgecolor='#004085', lw=2))
ax.annotate('', xy=(5, 4.35), xytext=(5, 4.85), arrowprops=arrow_props)
ax.text(5, 4.0, 'Synthetic prediction labels\n(5 scenarios × pre-specified sensitivity/specificity)', ha='center', va='center',
        fontsize=10, bbox=box_style)
for xpos in [1.8, 5.0, 8.2]:
    ax.annotate('', xy=(xpos, 2.85), xytext=(5, 3.55), arrowprops=arrow_props)
for xpos, ypos, label, ec, fc in [
    (1.8, 2.5, 'Radiologist GT',    '#2c7bb6', '#d0e8f8'),
    (5.0, 2.5, 'CT-RATE original',  '#d7191c', '#fcd5d5'),
    (8.2, 2.5, 'LLM majority vote', '#1a9641', '#d5f5d8'),
]:
    ax.text(xpos, ypos, label, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=fc, edgecolor=ec, lw=1.5))
for xpos in [1.8, 5.0, 8.2]:
    ax.annotate('', xy=(xpos, 1.65), xytext=(xpos, 2.15), arrowprops=arrow_props)
ax.text(5, 1.3, 'Performance metrics (F1, balanced accuracy, sensitivity, specificity, PPV, NPV, accuracy)', ha='center', va='center',
        fontsize=9.5, bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='#555', lw=1))
ax.annotate('', xy=(5, 0.65), xytext=(5, 1.0), arrowprops=arrow_props)
ax.text(5, 0.35, 'Absolute deviation from radiologist-GT-based estimate\n(report-level bootstrap 95% CI, B = 1000)', ha='center', va='center',
        fontsize=9.5, bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff9c4', edgecolor='#f57f17', lw=1.5))

ax.set_title('Simulation design', fontsize=13, fontweight='bold', pad=8)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'sim_fig2a_schematic.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: sim_fig2a_schematic.png')

### Figure 2B: Absolute Deviation Bar Plot (Main — F1 Score)

In [ ]:
def plot_deviation_bars(metric: str, title_suffix: str, fname: str):
    scenario_order = list(SCENARIOS.keys())
    x     = np.arange(len(scenario_order))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))

    all_hi = []
    for i, (ref, label, color) in enumerate([
        ('ct_rate',  'CT-RATE (original)',  REF_COLORS['ct_rate']),
        ('llm_vote', 'LLM majority vote',   REF_COLORS['llm_vote']),
    ]):
        offset = (i - 0.5) * width
        errs, lowers, uppers = [], [], []
        for sc in scenario_order:
            r = dev_bs_df[(dev_bs_df['scenario'] == sc) & (dev_bs_df['metric'] == metric)]
            if len(r) == 0:
                errs.append(np.nan); lowers.append(np.nan); uppers.append(np.nan)
                continue
            row = r.iloc[0]
            val = row[f'{ref}_error']
            hi  = row[f'{ref}_error_hi']
            errs.append(val)
            lowers.append(val - row[f'{ref}_error_lo'])
            uppers.append(hi - val)
            all_hi.append(hi)

        ax.bar(x + offset, errs, width=width, label=label, color=color,
               alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.errorbar(x + offset, errs, yerr=[lowers, uppers],
                    fmt='none', color='black', capsize=4, linewidth=1.2)

    ax.set_xticks(x)
    ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenario_order], fontsize=9)
    ax.set_ylabel(f'Absolute deviation in {title_suffix}', fontsize=11)
    ax.set_title(f'Absolute deviation from radiologist-GT-based estimate — {title_suffix}', fontsize=11)
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(all_hi) * 1.25 if all_hi else 0.3)

    fig.tight_layout()
    plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved: {fname}')


plot_deviation_bars('F1', 'F1 score', 'sim_fig2b_deviation_f1.png')

### Supplementary Figures: Balanced Accuracy, Sensitivity, Specificity Deviation

In [ ]:
plot_deviation_bars('Balanced_accuracy', 'Balanced accuracy', 'sim_sfig1_deviation_balanced_acc.png')
plot_deviation_bars('Sensitivity',       'Sensitivity',        'sim_sfig2_deviation_sensitivity.png')
plot_deviation_bars('Specificity',       'Specificity',        'sim_sfig3_deviation_specificity.png')

### CT-RATE vs LLM Vote: Direct Performance Difference

`metric(pred, CT-RATE) − metric(pred, LLM vote)` の bootstrap 分布。
正 = CT-RATE を reference にした場合に性能が高く見える、負 = LLM vote の方が高く見える。

In [ ]:
def plot_ct_vs_llm(src_df, focus_metrics, title, fname, ylim=(-0.45, 0.45)):
    scenario_order = list(SCENARIOS.keys())
    x     = np.arange(len(scenario_order))
    width = 0.55

    n_metrics = len(focus_metrics)
    ncols = 2
    nrows = (n_metrics + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 4))
    axes = axes.flatten() if n_metrics > 1 else [axes]

    for ax, (metric, metric_label) in zip(axes, focus_metrics):
        vals, lowers, uppers = [], [], []
        for sc in scenario_order:
            r = src_df[(src_df["scenario"] == sc) & (src_df["metric"] == metric)].iloc[0]
            v  = r["ct_vs_llm"]
            lo = r["ct_vs_llm_lo"]
            hi = r["ct_vs_llm_hi"]
            vals.append(v)
            lowers.append(v - lo)
            uppers.append(hi - v)

        colors = ["#d7191c" if v >= 0 else "#1a9641" for v in vals]
        ax.bar(x, vals, width=width, color=colors, alpha=0.80, edgecolor="white", linewidth=0.5)
        ax.errorbar(x, vals, yerr=[lowers, uppers],
                    fmt="none", color="black", capsize=4, linewidth=1.2)
        ax.axhline(0, color="grey", linewidth=1.0, linestyle="--")
        ax.set_ylim(ylim)

        ax.set_xticks(x)
        ax.set_xticklabels([SCENARIO_SHORT[s] for s in scenario_order], fontsize=9, rotation=20, ha="right")
        ax.set_ylabel("CT-RATE - LLM vote (" + metric_label + ")", fontsize=9)
        ax.set_title(metric_label, fontsize=10)
        ax.yaxis.grid(True, linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.legend(
            handles=[
                mpatches.Patch(color="#d7191c", alpha=0.8, label="CT-RATE higher"),
                mpatches.Patch(color="#1a9641", alpha=0.8, label="LLM vote higher"),
            ],
            fontsize=8, loc="upper right"
        )

    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.savefig(FIGURES_DIR / fname, bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved: {fname}")


plot_ct_vs_llm(
    src_df=dev_bs_df,
    focus_metrics=[
        ("F1",                "F1 score"),
        ("Balanced_accuracy", "Balanced accuracy"),
        ("Sensitivity",       "Sensitivity"),
        ("Specificity",       "Specificity"),
    ],
    title="CT-RATE vs LLM majority vote -- direct performance difference (pooled)",
    fname="sim_fig_ct_vs_llm_pooled.png",
)


### Combined Two-Panel Figure (2A + 2B)

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.08)

# --- Panel A: schematic ---
ax_a = fig.add_subplot(gs[0])
ax_a.set_xlim(0, 10)
ax_a.set_ylim(0, 6)
ax_a.axis('off')

box_style_a   = dict(boxstyle='round,pad=0.5', facecolor='#ddeeff', edgecolor='#336699', linewidth=1.5)
arrow_props_a = dict(arrowstyle='->', color='#336699', lw=1.5)

ax_a.text(5, 5.3, 'Radiologist-annotated reference set\n(100 reports × 18 labels)',
          ha='center', va='center', fontsize=9,
          bbox=dict(boxstyle='round,pad=0.5', facecolor='#cce5ff', edgecolor='#004085', lw=2))
ax_a.annotate('', xy=(5, 4.45), xytext=(5, 4.95), arrowprops=arrow_props_a)
ax_a.text(5, 4.1, 'Synthetic prediction labels\n(5 scenarios × pre-specified\nsensitivity / specificity)',
          ha='center', va='center', fontsize=9, bbox=box_style_a)
for xpos in [1.8, 5.0, 8.2]:
    ax_a.annotate('', xy=(xpos, 2.85), xytext=(5, 3.6), arrowprops=arrow_props_a)
for xpos, label, ec, fc in [
    (1.8, 'Radiologist GT',    '#2c7bb6', '#d0e8f8'),
    (5.0, 'CT-RATE\noriginal', '#d7191c', '#fcd5d5'),
    (8.2, 'LLM majority\nvote','#1a9641', '#d5f5d8'),
]:
    ax_a.text(xpos, 2.5, label, ha='center', va='center', fontsize=8.5,
              bbox=dict(boxstyle='round,pad=0.4', facecolor=fc, edgecolor=ec, lw=1.5))
for xpos in [1.8, 5.0, 8.2]:
    ax_a.annotate('', xy=(xpos, 1.7), xytext=(xpos, 2.15), arrowprops=arrow_props_a)
ax_a.text(5, 1.35, 'Performance metrics\n(F1, balanced accuracy, sensitivity,\nspecificity, PPV, NPV)',
          ha='center', va='center', fontsize=8.5,
          bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='#555', lw=1))
ax_a.annotate('', xy=(5, 0.65), xytext=(5, 1.0), arrowprops=arrow_props_a)
ax_a.text(5, 0.35, 'Absolute deviation from\nradiologist-GT-based estimate',
          ha='center', va='center', fontsize=8.5,
          bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff9c4', edgecolor='#f57f17', lw=1.5))
ax_a.set_title('Simulation design', fontsize=11, loc='left', pad=6)

# --- Panel B: deviation bar plot (F1) ---
ax_b = fig.add_subplot(gs[1])

scenario_order = list(SCENARIOS.keys())
x     = np.arange(len(scenario_order))
width = 0.35
all_hi_b = []

for i, (ref, label, color) in enumerate([
    ('ct_rate',  'CT-RATE (original)', REF_COLORS['ct_rate']),
    ('llm_vote', 'LLM majority vote',  REF_COLORS['llm_vote']),
]):
    offset = (i - 0.5) * width
    errs, lowers, uppers = [], [], []
    for sc in scenario_order:
        r = dev_bs_df[(dev_bs_df['scenario'] == sc) & (dev_bs_df['metric'] == 'F1')].iloc[0]
        val = r[f'{ref}_error']
        hi  = r[f'{ref}_error_hi']
        errs.append(val)
        lowers.append(val - r[f'{ref}_error_lo'])
        uppers.append(hi - val)
        all_hi_b.append(hi)
    ax_b.bar(x + offset, errs, width=width, label=label, color=color,
             alpha=0.85, edgecolor='white', linewidth=0.5)
    ax_b.errorbar(x + offset, errs, yerr=[lowers, uppers],
                  fmt='none', color='black', capsize=4, linewidth=1.2)

ax_b.set_xticks(x)
ax_b.set_xticklabels([SCENARIO_LABELS[s] for s in scenario_order], fontsize=8)
ax_b.set_ylabel('Absolute deviation in F1 score', fontsize=10)
ax_b.set_title('Absolute deviation from radiologist-GT-based F1 estimate', fontsize=10, loc='left')
ax_b.legend(fontsize=9)
ax_b.yaxis.grid(True, linestyle='--', alpha=0.4)
ax_b.set_axisbelow(True)
ax_b.set_ylim(0, max(all_hi_b) * 1.25)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'sim_fig2_combined.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: sim_fig2_combined.png')

## 8. Supplementary: Per-Label Error Reduction

Label-wise analysis: compute error reduction per label across all scenarios.

In [ ]:
labelwise_rows = []

for scenario in SCENARIOS:
    pred_col = f'pred_{scenario}'
    for lbl in LABEL_COLS:
        sub = long_df[long_df['label_name'] == lbl]
        y_pred = sub[pred_col].astype(int).values

        results_lbl = {}
        for ref in REFERENCES:
            y_ref = sub[ref].astype(int).values
            m = compute_metrics_from_cm(y_ref, y_pred)
            for mn, mv in m.items():
                results_lbl[f'{ref}__{mn}'] = mv

        for metric in METRICS:
            gt_val  = results_lbl[f'radiologist_gt__{metric}']
            ct_val  = results_lbl[f'ct_rate__{metric}']
            llm_val = results_lbl[f'llm_vote__{metric}']

            ct_err  = abs(ct_val  - gt_val) if not np.isnan(ct_val)  else np.nan
            llm_err = abs(llm_val - gt_val) if not np.isnan(llm_val) else np.nan
            red     = (ct_err - llm_err) if (not np.isnan(ct_err) and not np.isnan(llm_err)) else np.nan

            labelwise_rows.append({
                'scenario':       scenario,
                'label_name':     lbl,
                'metric':         metric,
                'gt_estimate':    gt_val,
                'ct_rate_est':    ct_val,
                'llm_vote_est':   llm_val,
                'ct_rate_error':  ct_err,
                'llm_vote_error': llm_err,
                'error_reduction': red,
            })

labelwise_df = pd.DataFrame(labelwise_rows)
labelwise_df.to_csv(TABLES_DIR / 'sim_labelwise_error_reduction.csv', index=False)
print('Saved: sim_labelwise_error_reduction.csv')
print()
print('Mean error reduction per label (F1, averaged across scenarios):')
mean_red = (
    labelwise_df[labelwise_df['metric'] == 'F1']
    .groupby('label_name')['error_reduction']
    .mean()
    .sort_values(ascending=False)
    .round(4)
)
print(mean_red.to_string())

In [ ]:
# Supplementary Figure: per-label error reduction heatmap (F1, all scenarios)
for metric in ['F1', 'Balanced_accuracy']:
    sub = labelwise_df[labelwise_df['metric'] == metric]
    pivot = sub.pivot_table(index='label_name', columns='scenario', values='error_reduction')
    pivot = pivot[list(SCENARIOS.keys())]
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    annot = pivot.map(lambda v: f'{v:.3f}' if not np.isnan(v) else '')

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        pivot.astype(float),
        annot=annot, fmt='', cmap='RdYlGn',
        center=0, linewidths=0.4, linecolor='white',
        annot_kws={'size': 8},
        cbar_kws={'label': 'Error reduction\n(CT-RATE error − LLM vote error)', 'shrink': 0.7},
        ax=ax,
    )
    ax.set_title(
        f'Per-label error reduction in {metric}\n'
        f'(positive = LLM vote closer to radiologist GT)',
        pad=10
    )
    ax.set_xticklabels(
        [SCENARIO_LABELS[s].replace('\n', ' ') for s in list(SCENARIOS.keys())],
        rotation=20, ha='right', fontsize=9
    )
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')
    fig.tight_layout()
    fname = f'sim_sfig4_labelwise_error_reduction_{metric.lower()}.png'
    plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved: {fname}')

## 9. Focus Analysis: Lymphadenopathy

Lymphadenopathy は label agreement 解析でも大きな CT-RATE vs radiologist GT 乖離が確認されたラベル。
ここでは Lymphadenopathy 単独に絞った simulation 分析を行う。

- **Label-specific bootstrap**: 100 reports を復元抽出し、Lymphadenopathy の行 (100 行) だけで metric を計算。
- **Point estimates + bootstrap 95% CI** を pooled 全体と並べて比較する。

In [ ]:
FOCUS_LABEL = 'Lymphadenopathy'

lad = long_df[long_df['label_name'] == FOCUS_LABEL].copy()

print(f'=== {FOCUS_LABEL}: label distribution ===')
print(f'  N reports           : {len(lad)}')
print(f'  Radiologist GT pos  : {lad["radiologist_gt"].sum()} / {len(lad)}  ({lad["radiologist_gt"].mean():.1%})')
print(f'  CT-RATE pos         : {lad["ct_rate"].sum()} / {len(lad)}  ({lad["ct_rate"].mean():.1%})')
print(f'  LLM majority pos    : {lad["llm_vote"].sum()} / {len(lad)}  ({lad["llm_vote"].mean():.1%})')
print()

# Confusion matrices of CT-RATE and LLM vote vs radiologist GT
for ref, name in [('ct_rate', 'CT-RATE'), ('llm_vote', 'LLM majority vote')]:
    cm = confusion_matrix(lad['radiologist_gt'], lad[ref], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    f1v  = f1_score(lad['radiologist_gt'], lad[ref], average='binary', zero_division=0)
    print(f'  {name} vs Radiologist GT:')
    print(f'    TP={tp} FP={fp} FN={fn} TN={tn}')
    print(f'    Sensitivity={sens:.3f}  Specificity={spec:.3f}  PPV={ppv:.3f}  F1={f1v:.3f}')
    print()

# Point estimates for each scenario (Lymphadenopathy only)
lad_pt_rows = []
for scenario in SCENARIOS:
    res = evaluate_scenario_on_df(lad, scenario)
    for ref in REFERENCES:
        for metric in METRICS:
            lad_pt_rows.append({
                'scenario':  scenario,
                'reference': ref,
                'metric':    metric,
                'estimate':  res[f'{ref}__{metric}'],
            })

lad_pt_df = pd.DataFrame(lad_pt_rows)

gt_pt = lad_pt_df[lad_pt_df['reference'] == 'radiologist_gt'][['scenario', 'metric', 'estimate']].rename(columns={'estimate': 'gt_estimate'})
lad_pt_df = lad_pt_df.merge(gt_pt, on=['scenario', 'metric'])
lad_pt_df['abs_deviation'] = (lad_pt_df['estimate'] - lad_pt_df['gt_estimate']).abs()

print('Point estimates (F1) — Lymphadenopathy:')
pivot_lad = lad_pt_df[lad_pt_df['metric'] == 'F1'].pivot_table(
    index='scenario', columns='reference', values='estimate'
)[REFERENCES]
pivot_lad.columns = [REF_LABELS[r] for r in REFERENCES]
print(pivot_lad.round(4).to_string())

In [ ]:
# Label-specific bootstrap for Lymphadenopathy
# Each iteration: sample 100 reports → take only Lymphadenopathy rows → 100 rows

rng_lad = np.random.default_rng(RANDOM_SEED)

lad_bs_store = {
    scenario: {ref: {metric: [] for metric in METRICS} for ref in REFERENCES}
    for scenario in SCENARIOS
}

# Pre-index long_df by report_id for fast lookup
lad_by_report = {
    rep: long_df[(long_df['report_id'] == rep) & (long_df['label_name'] == FOCUS_LABEL)]
    for rep in SORTED_IDX
}

for b in range(B):
    sampled_reports = rng_lad.choice(report_ids, size=N_reports, replace=True)
    report_counts   = pd.Series(sampled_reports).value_counts()

    sub_rows = []
    for rep_id, count in report_counts.items():
        for _ in range(count):
            sub_rows.append(lad_by_report[rep_id])
    sub_df = pd.concat(sub_rows, ignore_index=True)

    for scenario in SCENARIOS:
        res = evaluate_scenario_on_df(sub_df, scenario)
        for ref in REFERENCES:
            for metric in METRICS:
                lad_bs_store[scenario][ref][metric].append(res[f'{ref}__{metric}'])

print(f'Lymphadenopathy label-specific bootstrap complete. B={B}')

# Assemble bootstrap CI table
lad_bs_rows = []
for scenario in SCENARIOS:
    for ref in REFERENCES:
        for metric in METRICS:
            vals = np.array(lad_bs_store[scenario][ref][metric])
            lad_bs_rows.append({
                'scenario':        scenario,
                'reference':       ref,
                'metric':          metric,
                'bootstrap_lower': float(np.nanpercentile(vals, 2.5)),
                'bootstrap_upper': float(np.nanpercentile(vals, 97.5)),
            })

lad_bs_df = pd.DataFrame(lad_bs_rows)

lad_results_df = lad_pt_df.merge(
    lad_bs_df[['scenario', 'reference', 'metric', 'bootstrap_lower', 'bootstrap_upper']],
    on=['scenario', 'reference', 'metric']
)

print()
print('Bootstrap CIs computed. F1 for high_balanced:')
sub = lad_results_df[
    (lad_results_df['metric'] == 'F1') & (lad_results_df['scenario'] == 'high_balanced')
][['reference', 'estimate', 'bootstrap_lower', 'bootstrap_upper']]
sub['reference'] = sub['reference'].map(REF_LABELS)
print(sub.to_string(index=False))

In [ ]:
# Deviation table for Lymphadenopathy with label-specific bootstrap CIs
lad_dev_rows = []
for scenario in SCENARIOS:
    for metric in METRICS:
        gt_bs  = np.array(lad_bs_store[scenario]['radiologist_gt'][metric])
        ct_bs  = np.array(lad_bs_store[scenario]['ct_rate'][metric])
        llm_bs = np.array(lad_bs_store[scenario]['llm_vote'][metric])

        ct_err_bs  = np.abs(ct_bs  - gt_bs)
        llm_err_bs = np.abs(llm_bs - gt_bs)
        red_bs     = ct_err_bs - llm_err_bs
        diff_bs    = ct_bs - llm_bs   # direct CT-RATE minus LLM vote

        pt_ct  = lad_pt_df[(lad_pt_df['scenario'] == scenario) & (lad_pt_df['metric'] == metric) & (lad_pt_df['reference'] == 'ct_rate')]
        pt_llm = lad_pt_df[(lad_pt_df['scenario'] == scenario) & (lad_pt_df['metric'] == metric) & (lad_pt_df['reference'] == 'llm_vote')]
        ct_dev  = float(pt_ct['abs_deviation'].values[0])  if len(pt_ct)  > 0 else np.nan
        llm_dev = float(pt_llm['abs_deviation'].values[0]) if len(pt_llm) > 0 else np.nan
        ct_est_pt  = float(pt_ct['estimate'].values[0])  if len(pt_ct)  > 0 else np.nan
        llm_est_pt = float(pt_llm['estimate'].values[0]) if len(pt_llm) > 0 else np.nan

        lad_dev_rows.append({
            'scenario':           scenario,
            'metric':             metric,
            'ct_rate_error':      ct_dev,
            'ct_rate_error_lo':   float(np.nanpercentile(ct_err_bs, 2.5)),
            'ct_rate_error_hi':   float(np.nanpercentile(ct_err_bs, 97.5)),
            'llm_vote_error':     llm_dev,
            'llm_vote_error_lo':  float(np.nanpercentile(llm_err_bs, 2.5)),
            'llm_vote_error_hi':  float(np.nanpercentile(llm_err_bs, 97.5)),
            'error_reduction':    ct_dev - llm_dev if not (np.isnan(ct_dev) or np.isnan(llm_dev)) else np.nan,
            'error_reduction_lo': float(np.nanpercentile(red_bs, 2.5)),
            'error_reduction_hi': float(np.nanpercentile(red_bs, 97.5)),
            'ct_vs_llm':          ct_est_pt - llm_est_pt if not (np.isnan(ct_est_pt) or np.isnan(llm_est_pt)) else np.nan,
            'ct_vs_llm_lo':       float(np.nanpercentile(diff_bs, 2.5)),
            'ct_vs_llm_hi':       float(np.nanpercentile(diff_bs, 97.5)),
        })

lad_dev_df = pd.DataFrame(lad_dev_rows)

print(f'=== {FOCUS_LABEL}: Deviation from radiologist-GT-based estimates ===')
print()
for metric in ['F1', 'Balanced_accuracy']:
    sub = lad_dev_df[lad_dev_df['metric'] == metric][
        ['scenario', 'ct_rate_error', 'llm_vote_error', 'error_reduction',
         'ct_vs_llm', 'ct_vs_llm_lo', 'ct_vs_llm_hi']
    ].round(4)
    print(f'Metric: {metric}')
    print(sub.to_string(index=False))
    print()

lad_dev_df.to_csv(TABLES_DIR / 'sim_lymphadenopathy_deviation.csv', index=False)
print('Saved: sim_lymphadenopathy_deviation.csv')

In [ ]:
# Figure: Lymphadenopathy — absolute deviation (4-panel)
focus_metrics = [
    ('F1',                'F1 score'),
    ('Balanced_accuracy', 'Balanced accuracy'),
    ('Sensitivity',       'Sensitivity'),
    ('Specificity',       'Specificity'),
]

scenario_order = list(SCENARIOS.keys())
x     = np.arange(len(scenario_order))
width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, (metric, metric_label) in zip(axes, focus_metrics):
    all_hi = []
    for i, (ref, label, color) in enumerate([
        ('ct_rate',  'CT-RATE (original)', REF_COLORS['ct_rate']),
        ('llm_vote', 'LLM majority vote',  REF_COLORS['llm_vote']),
    ]):
        errs, lowers, uppers = [], [], []
        for sc in scenario_order:
            r = lad_dev_df[(lad_dev_df['scenario'] == sc) & (lad_dev_df['metric'] == metric)].iloc[0]
            val = r[f'{ref}_error']
            hi  = r[f'{ref}_error_hi']
            errs.append(val)
            lowers.append(val - r[f'{ref}_error_lo'])
            uppers.append(hi - val)
            all_hi.append(hi)

        offset = (i - 0.5) * width
        ax.bar(x + offset, errs, width=width, label=label, color=color,
               alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.errorbar(x + offset, errs, yerr=[lowers, uppers],
                    fmt='none', color='black', capsize=3, linewidth=1.1)

    ax.set_xticks(x)
    ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenario_order], fontsize=7.5)
    ax.set_ylabel(f'Absolute deviation in\n{metric_label}', fontsize=9)
    ax.set_title(metric_label, fontsize=10)
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(all_hi) * 1.25 if all_hi else 0.3)

fig.suptitle(f'Absolute deviation from radiologist-GT-based estimates — {FOCUS_LABEL}', fontsize=11)
fig.tight_layout()
fname = 'sim_sfig_lymphadenopathy_deviation.png'
plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
plt.show()
print(f'Saved: {fname}')

# Figure: Lymphadenopathy — direct CT-RATE vs LLM comparison
plot_ct_vs_llm(
    src_df=lad_dev_df,
    focus_metrics=focus_metrics,
    title=f'CT-RATE vs LLM majority vote — direct performance difference ({FOCUS_LABEL})',
    fname='sim_sfig_lymphadenopathy_ct_vs_llm.png',
)

In [ ]:
# Pooled vs Lymphadenopathy comparison — F1 error side by side
print('=== Comparison: Pooled (all 18 labels) vs Lymphadenopathy only ===')
print()
print('Metric: F1 — CT-RATE error  /  LLM vote error  /  Error reduction')
print()
for scenario in SCENARIOS:
    p = dev_bs_df[(dev_bs_df['scenario'] == scenario) & (dev_bs_df['metric'] == 'F1')].iloc[0]
    l = lad_dev_df[(lad_dev_df['scenario'] == scenario) & (lad_dev_df['metric'] == 'F1')].iloc[0]
    print(f'  {scenario}:')
    print(f'    Pooled  : CT-RATE={p["ct_rate_error"]:.4f}  LLM={p["llm_vote_error"]:.4f}  reduction={p["error_reduction"]:.4f}  95%CI [{p["error_reduction_lo"]:.4f}, {p["error_reduction_hi"]:.4f}]')
    print(f'    Lymphad.: CT-RATE={l["ct_rate_error"]:.4f}  LLM={l["llm_vote_error"]:.4f}  reduction={l["error_reduction"]:.4f}  95%CI [{l["error_reduction_lo"]:.4f}, {l["error_reduction_hi"]:.4f}]')
    print()

In [ ]:
# Summary figure: 4x2 grid (All labels top row, Lymphadenopathy bottom row)
focus_metrics_summary = [
    ('F1',                'F1 score'),
    ('Balanced_accuracy', 'Balanced accuracy'),
    ('Sensitivity',       'Sensitivity'),
    ('Specificity',       'Specificity'),
]
scenario_order = list(SCENARIOS.keys())
x     = np.arange(len(scenario_order))
width = 0.55

# Per-row, per-column ylim
ylim_grid = {
    0: {0: (-0.1, 0.1), 1: (-0.1, 0.1), 2: (-0.1, 0.1), 3: (-0.1, 0.1)},
    1: {0: (-0.3, 0.3), 1: (-0.3, 0.3), 2: (-0.5, 0.5), 3: (-0.5, 0.5)},
}

row_configs = [
    (dev_bs_df,  'All labels (pooled)'),
    (lad_dev_df, 'Lymphadenopathy'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for row_idx, (src_df, row_label) in enumerate(row_configs):
    for col_idx, (metric, metric_label) in enumerate(focus_metrics_summary):
        ax = axes[row_idx, col_idx]

        vals, lowers, uppers = [], [], []
        for sc in scenario_order:
            r = src_df[(src_df['scenario'] == sc) & (src_df['metric'] == metric)].iloc[0]
            v  = r['ct_vs_llm']
            lo = r['ct_vs_llm_lo']
            hi = r['ct_vs_llm_hi']
            vals.append(v)
            lowers.append(v - lo)
            uppers.append(hi - v)

        colors = ['#d7191c' if v >= 0 else '#1a9641' for v in vals]
        ax.bar(x, vals, width=width, color=colors, alpha=0.80, edgecolor='white', linewidth=0.5)
        ax.errorbar(x, vals, yerr=[lowers, uppers],
                    fmt='none', color='black', capsize=4, linewidth=1.2)
        ax.axhline(0, color='grey', linewidth=1.0, linestyle='--')
        ax.set_ylim(ylim_grid[row_idx][col_idx])
        ax.set_xticks(x)
        ax.set_xticklabels([SCENARIO_SHORT[s] for s in scenario_order],
                           fontsize=8, rotation=20, ha='right')
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)

        if row_idx == 0:
            ax.set_title(metric_label, fontsize=10)

        if col_idx == 0:
            ax.set_ylabel(row_label + '\nCT-RATE - LLM vote', fontsize=9)
        else:
            ax.set_ylabel('')

legend_handles = [
    mpatches.Patch(color='#d7191c', alpha=0.8, label='CT-RATE higher'),
    mpatches.Patch(color='#1a9641', alpha=0.8, label='LLM vote higher'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('CT-RATE vs LLM majority vote -- direct performance difference', fontsize=12)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(FIGURES_DIR / 'sim_fig_ct_vs_llm_summary_4x2.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: sim_fig_ct_vs_llm_summary_4x2.png')

In [ ]:
# Summary figure: 6x2 grid — adds PPV and NPV
focus_metrics_summary6 = [
    ('F1',                'F1 score'),
    ('Balanced_accuracy', 'Balanced accuracy'),
    ('Sensitivity',       'Sensitivity'),
    ('Specificity',       'Specificity'),
    ('PPV',               'PPV'),
    ('NPV',               'NPV'),
]
scenario_order = list(SCENARIOS.keys())
x     = np.arange(len(scenario_order))
width = 0.55

# Per-row, per-column ylim
ylim_grid6 = {
    0: {0: (-0.1, 0.1), 1: (-0.1, 0.1), 2: (-0.1, 0.1), 3: (-0.1, 0.1), 4: (-0.1, 0.1), 5: (-0.1, 0.1)},
    1: {0: (-0.3, 0.3), 1: (-0.3, 0.3), 2: (-0.5, 0.5), 3: (-0.5, 0.5), 4: (-0.5, 0.5), 5: (-0.5, 0.5)},
}

row_configs = [
    (dev_bs_df,  'All labels (pooled)'),
    (lad_dev_df, 'Lymphadenopathy'),
]

fig, axes = plt.subplots(2, 6, figsize=(26, 9))

for row_idx, (src_df, row_label) in enumerate(row_configs):
    for col_idx, (metric, metric_label) in enumerate(focus_metrics_summary6):
        ax = axes[row_idx, col_idx]

        vals, lowers, uppers = [], [], []
        for sc in scenario_order:
            r = src_df[(src_df['scenario'] == sc) & (src_df['metric'] == metric)].iloc[0]
            v  = r['ct_vs_llm']
            lo = r['ct_vs_llm_lo']
            hi = r['ct_vs_llm_hi']
            vals.append(v)
            lowers.append(v - lo)
            uppers.append(hi - v)

        colors = ['#d7191c' if v >= 0 else '#1a9641' for v in vals]
        ax.bar(x, vals, width=width, color=colors, alpha=0.80, edgecolor='white', linewidth=0.5)
        ax.errorbar(x, vals, yerr=[lowers, uppers],
                    fmt='none', color='black', capsize=4, linewidth=1.2)
        ax.axhline(0, color='grey', linewidth=1.0, linestyle='--')
        ax.set_ylim(ylim_grid6[row_idx][col_idx])
        ax.set_xticks(x)
        ax.set_xticklabels([SCENARIO_SHORT[s] for s in scenario_order],
                           fontsize=8, rotation=20, ha='right')
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)

        if row_idx == 0:
            ax.set_title(metric_label, fontsize=10)

        if col_idx == 0:
            ax.set_ylabel(row_label + '\nCT-RATE - LLM vote', fontsize=9)
        else:
            ax.set_ylabel('')

legend_handles = [
    mpatches.Patch(color='#d7191c', alpha=0.8, label='CT-RATE higher'),
    mpatches.Patch(color='#1a9641', alpha=0.8, label='LLM vote higher'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('CT-RATE vs LLM majority vote -- direct performance difference (with PPV/NPV)', fontsize=12)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(FIGURES_DIR / 'sim_fig_ct_vs_llm_summary_6x2.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: sim_fig_ct_vs_llm_summary_6x2.png')

## 9. Methods Text (English Draft)

> To examine the effect of label refinement on downstream model-performance estimation, we conducted a simulation analysis across all 18 abnormality categories using the radiologist-annotated reference set as the true ground truth. For each category, synthetic prediction labels were generated once to approximate prespecified sensitivity and specificity values against radiologist annotations and were then held fixed. Five performance scenarios were considered: high balanced performance (sensitivity = 0.90, specificity = 0.90), high sensitivity (0.95/0.85), high specificity (0.85/0.95), moderate balanced performance (0.80/0.80), and weak balanced performance (0.70/0.70).
>
> The fixed synthetic predictions were evaluated using three reference standards: radiologist annotations (true ground truth), original CT-RATE labels, and multi-LLM majority-vote labels (positive if ≥2 of 3 models agreed). For each metric, we calculated the absolute deviation of CT-RATE–based and majority-vote–based estimates from the radiologist-based estimate. Smaller deviations indicated more accurate estimation of downstream model performance. The primary outcome was the absolute deviation in F1 score; secondary outcomes included absolute deviations in balanced accuracy, sensitivity, and specificity.
>
> Uncertainty was assessed using report-level bootstrap resampling (B = 1,000). In each bootstrap iteration, 100 reports were sampled with replacement from the reference set, preserving all 18 abnormality labels within each sampled report. Bootstrap 95% confidence intervals were obtained from the 2.5th and 97.5th percentiles across iterations. All analyses were performed in Python using NumPy, pandas, and scikit-learn.

In [ ]:
# Final summary
print('=== Analysis complete ===')
print()
print('Outputs:')
print('  Tables:')
print('    sim_achieved_sens_spec.csv         — achieved sensitivity/specificity per scenario/label')
print('    sim_performance_estimates.csv      — performance estimates with 95% bootstrap CI')
print('    sim_deviation_table.csv            — deviation from radiologist-GT with 95% CI')
print('    sim_labelwise_error_reduction.csv  — per-label error reduction')
print()
print('  Figures:')
print('    sim_fig2a_schematic.png            — Panel A: simulation design')
print('    sim_fig2b_deviation_f1.png         — Panel B: F1 deviation bar plot')
print('    sim_fig2_combined.png              — Combined 2-panel figure')
print('    sim_sfig1_deviation_balanced_acc   — Supplementary: balanced accuracy')
print('    sim_sfig2_deviation_sensitivity    — Supplementary: sensitivity')
print('    sim_sfig3_deviation_specificity    — Supplementary: specificity')
print('    sim_sfig4_labelwise_error_*        — Supplementary: per-label heatmap')